# DeepSeek NExT-QA: missing control accuracies

Version: 2026-10-13-py311-arrow17-fix. Enable a GPU and Internet in Kaggle settings. This notebook embeds its code and reference IDs; no code dataset upload is needed. The setup creates a separate Python 3.11 environment because the original Transformers/tokenizers stack does not provide Python 3.13 wheels. NumPy 1.26.4 and PyArrow 17.0.0 are pinned together. The notebook kernel is unchanged. Initial setup downloads CUDA PyTorch wheels and can take several minutes. It generates answers only, without routing capture.

Start with `SAMPLES=2` or `32`; use `256` to fill the original paper table. Keep the same output directory to resume and reuse completed answers. The original 256-item donor ring is used even in a pilot.

Attach the original Experiment 2 manifest/checkpoint if available. Otherwise the notebook reconstructs the seeded manifest and verifies all question/video IDs against the saved TRC CSV. Reconstruction cannot independently verify the historical cross-clip donor order or checkpoint revision. Review those before combining results with the old TRC table.


In [ ]:
from pathlib import Path
import base64, io, json, os, subprocess, sys, zipfile
CODE = Path('/kaggle/working/deepseek_accuracy_code')
CODE.mkdir(parents=True, exist_ok=True)
PAYLOAD = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as archive:
    archive.extractall(CODE)
os.environ.update(USE_TORCH='1', USE_TF='0', USE_FLAX='0', TOKENIZERS_PARALLELISM='false')
subprocess.run([sys.executable, '-u', str(CODE/'setup_runtime.py')], check=True)
PYTHON = '/kaggle/working/deepseek_accuracy_py311/bin/python'


In [ ]:
# Use 2 for a smoke test, 32 for a pilot, 256 for paper results.
SAMPLES = 32
OUTPUT = Path('/kaggle/working/deepseek_control_accuracy')
MODEL = 'deepseek-ai/deepseek-vl2-tiny'  # Or the original uploaded checkpoint path.
REVISION = None  # Set to the original checkpoint revision if available.
ORIGINAL_MANIFEST = None  # Set to the original nextqa_256_seed20260819.json if available.
args = [PYTHON, '-u', str(CODE/'run.py'), '--limit', str(SAMPLES),
        '--model', MODEL, '--output-dir', str(OUTPUT)]
if REVISION:
    args += ['--revision', REVISION]
if ORIGINAL_MANIFEST:
    args += ['--manifest', str(ORIGINAL_MANIFEST)]


In [ ]:
subprocess.run(args, check=True)


In [ ]:
import pandas as pd
display(pd.read_csv(OUTPUT/'condition_accuracy.csv'))
print((OUTPUT/'checks.json').read_text())


## Download outputs
The ZIP contains per-condition answers, prompts, frame indices, donor IDs, runtime provenance and summary/check files. A pilot is exploratory; do not insert its accuracies into an n=256 table. Router permutation remains n/a because it changes routing labels only.


In [ ]:
import shutil
archive = shutil.make_archive('/kaggle/working/deepseek_control_accuracy_results', 'zip', OUTPUT)
print(archive)
from IPython.display import FileLink
display(FileLink(archive))
